In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report

def auto_ml_pipeline(file_path):
    # 1. 파일 확장자별 자동 로드
    if file_path.endswith('.csv'):
        df = pd.read_csv(file_path)
    elif file_path.endswith(('.xlsx', '.xls')):
        df = pd.read_excel(file_path)
    else:
        raise ValueError("지원하지 않는 파일 형식입니다. (.csv, .xlsx, .xls)")

    print(f" 파일 로드 완료: {file_path} (크기: {df.shape})")

    # 2. 결측치 전처리 (50% 이상 결측 열 삭제 & 중앙값 대치)
    df_cleaned = df.dropna(thresh=len(df) * 0.5, axis=1).copy()
    numeric_cols = df_cleaned.select_dtypes(include=['number']).columns
    df_cleaned[numeric_cols] = df_cleaned[numeric_cols].fillna(df_cleaned[numeric_cols].median())

    # 3. 타겟 열(불량 여부) 자동 감지
    # 고유값이 2개(이진 분류)인 열 후보 찾기
    candidate_targets = [col for col in df_cleaned.columns if df_cleaned[col].nunique() == 2]

    if not candidate_targets:
        raise ValueError("이진 분류(정상/불량)를 위한 타겟 열을 찾지 못했습니다.")

    # 마지막 열이거나 키워드(target, fail, pass, label 등)를 가진 열 우선 선택
    target_col = candidate_targets[-1]
    for col in candidate_targets:
        if any(keyword in col.lower() for keyword in ['pass', 'fail', 'target', 'label', 'defect', 'y']):
            target_col = col
            break

    print(f" 자동 감지된 타겟(불량) 열: '{target_col}'")

    # 4. 시간/ID/식별자 열 자동 판별 및 제거
    feature_cols = [c for c in df_cleaned.columns if c != target_col]
    drop_cols = []
    for c in feature_cols:
        # 문자열 데이터이거나 열 이름에 time/date/id 등이 들어간 경우 제거
        if df_cleaned[c].dtype == 'object' or any(k in c.lower() for k in ['time', 'date', 'id', 'wafer']):
            drop_cols.append(c)

    X = df_cleaned[feature_cols].drop(columns=drop_cols, errors='ignore')

    # 분산이 0인(값이 일정한) 센서 제거
    X = X.loc[:, X.std() > 0]
    y = df_cleaned[target_col]

    # 5. Top 10 센서 추출
    num_features = min(10, X.shape[1])
    selector = SelectKBest(score_func=f_classif, k=num_features)
    X_important = selector.fit_transform(X, y)
    selected_features = X.columns[selector.get_support()]
    print(f" 선택된 주요 센서 Top {num_features}:", list(selected_features))

    # 6. SMOTE + 랜덤포레스트 학습
    X_train, X_test, y_train, y_test = train_test_split(X_important, y, test_size=0.2, random_state=42, stratify=y)

    # 클래스 오버샘플링
    smote = SMOTE(random_state=42)
    X_train_over, y_train_over = smote.fit_resample(X_train, y_train)

    model = RandomForestClassifier(random_state=42)
    model.fit(X_train_over, y_train_over)

    # 7. 평가
    print("\n---  최종 모델 예측 성적표 ---")
    print(classification_report(y_test, model.predict(X_test)))

    return list(selected_features), target_col

# --------------------------------------------------
# 사용 예시: 파일 이름만 전달하면 바로 실행됩니다.
top_sensors, target = auto_ml_pipeline('uci-secom.csv')

 파일 로드 완료: uci-secom.csv (크기: (1567, 592))
 자동 감지된 타겟(불량) 열: 'Pass/Fail'
 선택된 주요 센서 Top 10: ['21', '28', '59', '103', '348', '430', '431', '434', '435', '510']

---  최종 모델 예측 성적표 ---
              precision    recall  f1-score   support

          -1       0.94      0.94      0.94       293
           1       0.19      0.19      0.19        21

    accuracy                           0.89       314
   macro avg       0.57      0.57      0.57       314
weighted avg       0.89      0.89      0.89       314

